# 07_attention: Dot-Product Scaling and Softmax Saturation
    
This notebook implements Scaled Dot-Product Attention calculations from scratch to demonstrate the variance scaling effect of dividing by $\sqrt{d_k}$.


In [1]:
import torch
import torch.nn.functional as F

# Sequence length L=4, dimension d_k=128
torch.manual_seed(42)
L, d_k = 4, 128

# Simulating Query, Key, and Value projections
Q = torch.randn(L, d_k)
K = torch.randn(L, d_k)
V = torch.randn(L, d_k)

# 1. Unscaled Attention
scores_unscaled = torch.matmul(Q, K.T)
weights_unscaled = F.softmax(scores_unscaled, dim=-1)

# 2. Scaled Attention (dividing by sqrt(d_k))
scaling_factor = d_k ** 0.5
scores_scaled = scores_unscaled / scaling_factor
weights_scaled = F.softmax(scores_scaled, dim=-1)

print("=== Unscaled Scores ===")
print(scores_unscaled)

print("\n=== Unscaled Attention Weights (Variance is high, scores saturate) ===")
print(weights_unscaled)
print("Unscaled weights variance:", torch.var(weights_unscaled).item())

print("\n=== Scaled Attention Weights (Variance is normalized, weights are sensitive) ===")
print(weights_scaled)
print("Scaled weights variance:", torch.var(weights_scaled).item())


=== Unscaled Scores ===
tensor([[ 7.0454e+00, -2.4135e+00, -6.3372e-04, -1.0523e+01],
        [-3.7554e+00, -8.5303e+00, -4.2125e+00, -1.1191e+01],
        [-3.3453e+00, -1.5157e+01, -1.3826e+01,  8.9298e+00],
        [ 7.7142e+00,  9.7219e+00,  5.1304e+00,  1.4382e+01]])

=== Unscaled Attention Weights (Variance is high, scores saturate) ===
tensor([[9.9905e-01, 7.7919e-05, 8.7003e-04, 2.3422e-08],
        [6.0897e-01, 5.1389e-03, 3.8553e-01, 3.5917e-04],
        [4.6666e-06, 3.4625e-11, 1.3096e-10, 1.0000e+00],
        [1.2570e-03, 9.3599e-03, 9.4884e-05, 9.8929e-01]])
Unscaled weights variance: 0.1664256602525711

=== Scaled Attention Weights (Variance is normalized, weights are sensitive) ===
tensor([[0.4584, 0.1987, 0.2459, 0.0970],
        [0.3190, 0.2092, 0.3064, 0.1654],
        [0.2124, 0.0748, 0.0841, 0.6287],
        [0.2086, 0.2492, 0.1660, 0.3762]])
Scaled weights variance: 0.020930269733071327


### Output Explanation
- Without scaling, large dot products push values into flat regions of the Softmax function, causing vanishing gradients.
- Dividing by $\sqrt{d_k}$ keeps variance constant, preserving model sensitivity during training.
